# 01. Khám phá dữ liệu đang có trong dự án
 Phân tích **ảnh tổng hợp có bbox đối chứng** và **bộ hồi quy phát triển có ảnh thật** riêng biệt.
 Đây không phải tập kiểm thử độc lập để công bố độ chính xác sản phẩm.

 Thư mục mẫu `data/raw/sample_detection` của notebook cũ không có trên máy này. Notebook hiện tại đọc `data/test_cases/ground_truth.json` và manifest ca khó đã đóng băng; không tạo lại ảnh/nhãn. Không suy loại phương tiện từ tên file.
 Chọn kernel `venv/Scripts/python.exe`, rồi **Run All**. Xem `notebooks/README.md` nếu thiếu dữ liệu.

In [ ]:
from pathlib import Path
import sys
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/core').is_dir() and (p / 'notebooks').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from notebooks._support import read_json, read_image, checked_crop, sha256
%matplotlib inline
plt.rcParams.update({'figure.figsize': (11, 4), 'font.size': 10})
print('Project:', ROOT)
print('Python:', sys.executable)

In [ ]:
from notebooks._support import load_synthetic_samples, load_development_cases, image_inventory

synthetic = load_synthetic_samples()
CASES_FILE = ROOT / 'data/recognition_regression/hardening_20261105/all_cases.json'
development = load_development_cases(CASES_FILE) if CASES_FILE.is_file() else []
print('Synthetic labels SHA256:', sha256(ROOT / 'data/test_cases/ground_truth.json'))
if development:
    print('Development manifest SHA256:', sha256(CASES_FILE))
else:
    print('CHƯA CÓ manifest hồi quy; chỉ phân tích bộ tổng hợp, không kết luận về ảnh thật.')
inventory = pd.concat([image_inventory(synthetic, 'synthetic_bbox'),
                       image_inventory(development, 'development_regression')], ignore_index=True)
display(inventory.groupby(['cohort', 'kind']).agg(
    images=('id', 'size'), unique_images=('sha256', 'nunique'), plates=('plates', 'sum')))
display(inventory[['cohort', 'id', 'width', 'height', 'plates', 'mean_gray']].head(10))
print('Số ảnh duy nhất theo SHA256 trên cả hai bộ:', inventory.sha256.nunique())
print('Không cộng số hàng thành số mẫu độc lập: có ảnh trùng giữa các bộ, và các khung hình cùng xe.')

## Kích thước và đặc trưng ảnh đo trực tiếp
 Độ sáng trung bình và phương sai Laplacian chỉ là mô tả ảnh; không phải điểm OCR.
 Nhóm tên `low_light` của bộ tổng hợp có thể chứa nhiễu màu mạnh, không đủ để đại diện ảnh chụp ban đêm.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
inventory.groupby(['cohort', 'kind']).size().plot.bar(ax=axes[0], rot=35, title='Số ảnh theo nguồn')
for kind, group in inventory.groupby('kind'):
    axes[1].scatter(group.width, group.height, label=kind, alpha=.65)
axes[1].set(xlabel='Width (px)', ylabel='Height (px)', title='Kích thước ảnh')
axes[1].legend()
for kind, group in inventory.groupby('kind'):
    axes[2].hist(group.mean_gray, bins=12, alpha=.5, label=kind)
axes[2].set(xlabel='Mean grayscale (0–255)', ylabel='Số ảnh', title='Phân bố độ sáng')
axes[2].legend()
plt.tight_layout()
plt.show()

## Kiểm tra nhãn và xem bbox
 Toàn bộ ảnh tổng hợp đã được giải mã, kiểm tra bbox hữu hạn, diện tích dương và nằm trong ảnh khi nạp.
 Kiểm tra tọa độ **không chứng minh** nhãn bao đúng biển: vẫn cần xem ảnh.
 Ảnh thật trong manifest chỉ có chuỗi đối chứng, chưa có bbox GT để tính AP.

In [ ]:
plate_rows = []
for sample in synthetic:
    image = read_image(sample['path'])
    for target in sample['truth']:
        crop = checked_crop(image, target['bbox'])
        plate_rows.append({'file': sample['file'], 'category': sample['category'],
                           'text': target['text'], 'width': crop.shape[1], 'height': crop.shape[0],
                           'ratio': crop.shape[1] / crop.shape[0]})
plates = pd.DataFrame(plate_rows)
display(plates.groupby('category').agg(plates=('text', 'size'),
                                       width_mean=('width', 'mean'), height_mean=('height', 'mean')))
selected = [next(s for s in synthetic if s['category'] == cat)
            for cat in sorted({s['category'] for s in synthetic})]
fig, axes = plt.subplots(1, len(selected), figsize=(16, 4), squeeze=False)
for ax, sample in zip(axes.flat, selected):
    image = read_image(sample['path'])
    for target in sample['truth']:
        x1, y1, x2, y2 = map(int, target['bbox'])
        cv2.rectangle(image, (x1,y1), (x2,y2), (0,180,0), 2)
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(sample['file'], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
print(f'Đã kiểm tra {len(synthetic)} ảnh tổng hợp, {len(plates)} bbox và {len(development)} ca hồi quy.')
print('Ảnh tổng hợp có nhãn phục vụ thử nghiệm; không thay thế ảnh giao thông thật.')
print('Ba ảnh parking_clip4 cùng xe có tương quan; SHA256 khác không đồng nghĩa mẫu độc lập.')
print('Cần thêm tập ảnh thật độc lập, bbox và chuỗi nhãn được kiểm tra thủ công trước khi nghiệm thu.')